In [29]:
from numpy.random.mtrand import f
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Callable, Iterator, Dict
from concurrent.futures import ProcessPoolExecutor
from scipy.special import lambertw
import multiprocessing
multiprocessing.set_start_method("fork", force=True)

# ANALYSIS FOR THE TWO ALLELE, TIME-INVARIANT FITNESS CASE

# global random number generator
rng = np.random.default_rng()


# wrightfishersampling simulates natural selection based on initial allele counts and a list of functions describing
# the fitness of each allele at each generation in time
def wrightfishersampling(K: np.ndarray, Ff: List[Callable[[int], float]]) -> Iterator[np.ndarray]:
    """
    params
    K = numpy array of initial allele count for each allele. for now, only 2 alleles
    Ff = Fitness function. change in F. list of lambda functions that change F
           based on the generation number for each allele.
           for now, fitness functions are constant (eg always return 1)
          input should be just the generation number (irrelevent here).
          output is fitness.

    yields np.ndarray: allele counts for current gen
    """

    # population number no longer needs to be an input because it is now implied
    N = np.sum(K)

    # input validation

    if np.any(K<=0):
      raise ValueError("Number of alleles cannot be negative")
    if N == 0:
        raise ValueError("error: population number is zero")
    if not isinstance(Ff, list) or not all(callable(x) for x in Ff):
        raise ValueError("Ff must be a list of callable functions")
    if not (K.size == len(Ff)):
        raise ValueError("K and Ff must have the same length")
    if K.size < 1:
        raise ValueError("K must be at least 1")

    t = 0  # we assume that the first generation is 0

    # while none of the alleles have yet fixed
    while True:

        yield K

        if np.max(K) == N:
            break

        # p is unnormalized initially because the equations in Ff model the fitness of each allele at each time
        # in relative terms rather than proportions to make it much easier to input
        F_values_at_t = np.array([func(t) for func in Ff],  dtype=float)
        p_unnormalized = K * F_values_at_t

        sum_p_unnormalized = np.sum(p_unnormalized)

        if sum_p_unnormalized == 0 or np.isnan(sum_p_unnormalized):
            pvals = np.zeros_like(K, dtype=float)
        else:
            # normalize
            pvals = p_unnormalized / sum_p_unnormalized
            # force the sum of pvals to be exactly 1.0 to prevent base 2 rounding errors
            pvals /= np.sum(pvals)

        # random multinomial is implemented to select the next generation using the pvals as likelihoods of
        # each allele.

        new_counts = rng.multinomial(n=N, pvals=pvals)
        K = new_counts

        t += 1  # next generation begins and the loop restarts

def run_single_simulation(initial_K_counts: np.ndarray, Ff: List[Callable[[int], float]]) -> Dict:
   """
   params
     initial_K_counts : np.ndarray, array representing the initial allele counts
     Ff : List[Callable[[int], float]], list of fitness functions for each allele

   returns
     dict containing:
       "fixed" bool indicating if an allele fixed
       "final_generations" total num of gens simulated
       "allele_counts_history" np.ndarray w/ allele counts per gen
       "allele_frequencies_history" np.ndarray w/ allele frequencies per gen
   """
   # get simulation history using vstack
   history = np.vstack(list(wrightfishersampling(initial_K_counts, Ff)))
   N = int(initial_K_counts.sum())

   final_t = history.shape[0]
   fixed = (history[-1] == N).any()
   # check if the first allele fixed
   first_allele_fixed = (history[-1][0] == N) if fixed else False

   allele_freq = history / N

   return {
       "fixed": fixed,
       "final_generations": final_t,
       "allele_counts_history": history,
       "allele_frequencies_history": allele_freq,
       "first_allele_fixed": first_allele_fixed
   }

def run_multiple_simulations_parallel(initial_K_counts: np.ndarray,
                                     Ff: List[Callable[[int], float]],
                                     num_simulations: int) -> List[Dict]:
   """
   runs multiple independent Wright–Fisher simulations in parallel at optimum efficiency

   params
     initial_K_counts : np.ndarray, initial allele counts
     Ff : List[Callable[[int], float]], list of fitness functions of each allele
     num_simulations : int

   output:e
     List[Dict]: list of dictionaries returned by run_single_simulation
   """
   with ProcessPoolExecutor() as executor:
       futures = [
           executor.submit(run_single_simulation, initial_K_counts, Ff)
           for _ in range(num_simulations)
       ]
       results = [future.result() for future in futures]
   return results


# define a named fitness function (can't use lambdas)
def constant_fitness_1(t: int) -> float:
   return 1.0

def constant_fitness_2(t: int) -> float:
   return 2.0

def constant_fitness_3(t: int) -> float:
   return 3.0

def constant_fitness_4(t: int) -> float:
   return 4.0

def constant_fitness_5(t: int) -> float:
   return 5.0

def constant_fitness_6(t: int) -> float:
   return 6.0

def constant_fitness_7(t: int) -> float:
   return 7.0

def constant_fitness_8(t: int) -> float:
   return 8.0

def constant_fitness_9(t: int) -> float:
   return 9.0

def constant_fitness_10(t: int) -> float:
   return 10.0

def constant_fitness_11(t: int) -> float:
   return 11.0

def constant_fitness_12(t: int) -> float:
   return 12.0

def Psi_s(s: float) -> float:
    res = 1 + lambertw(-(1 + s) * np.exp(-(1 + s))). real / (s + 1)
    # bug in lambertw(-пр.exp(-1)), k=-1) == -1
    if isinstance(res, np.ndarray):
        res[np.isnan(res)] = 0.
    else:
        if np.isnan(res):
         res = 0.
    return res

"""
import sympy as sp
Psi, s = sp.symbols('Psi s', real-True)
sp solve(sp.Eq(1 - Psi, sp.exp(-(1+s) * Psi)), Psi)
[s + LambertW((-s - 1)*exp(-s - 1)) + 1)/(s + 1)]
"""

def run_everything(INITIAL_ALLELE_COUNTS: np.ndarray, FITNESS_FUNCTIONS: List[Callable[[int], float]], NUM_RUNS: int):

    print(f"/n")
    print(f"INITIAL_ALLELE_COUNTS = {INITIAL_ALLELE_COUNTS}")
    print(f"FITNESS_FUNCTIONS = {FITNESS_FUNCTIONS}")

    fitness1 = FITNESS_FUNCTIONS[0](1)
    fitness2 = FITNESS_FUNCTIONS[1](1)
    size1 = INITIAL_ALLELE_COUNTS[0]
    size2 = INITIAL_ALLELE_COUNTS[1]

    # fitness1 = fitness2 * (1+s)
    s = fitness1 / fitness2 - 1

    if INITIAL_ALLELE_COUNTS[0] == 1 and s>= 0:

        if s == 0:
          expected_probability_of_first_allele_fixation = INITIAL_ALLELE_COUNTS[0] / (INITIAL_ALLELE_COUNTS[0] + INITIAL_ALLELE_COUNTS[1])
        elif s > 0:
          expected_probability_of_first_allele_fixation = Psi_s(s)

    else:
      w = fitness1 / fitness2
      wrightian_selection_coefficient = 1 - w
      malthusian_selection_coefficient = np.log(wrightian_selection_coefficient+1)
      malthusian_fitness = np.log(1+malthusian_selection_coefficient)
      if malthusian_fitness == 1:
        expected_probability_of_first_allele_fixation = (1-(1/malthusian_fitness)**INITIAL_ALLELE_COUNTS[0]) / (1-(1/malthusian_fitness)**(INITIAL_ALLELE_COUNTS[0]+INITIAL_ALLELE_COUNTS[1]))
      else:
        expected_probability_of_first_allele_fixation = 0

    # Now run the simulations
    sim_data = run_multiple_simulations_parallel(initial_K_counts=INITIAL_ALLELE_COUNTS,
                                        Ff=FITNESS_FUNCTIONS,
                                        num_simulations=NUM_RUNS)

    fixed_generations = [run['final_generations'] for run in sim_data if run['fixed']]
    first_allele_fixations = [run for run in sim_data if run['first_allele_fixed']]


    probability_of_first_allele_fixation = len(first_allele_fixations) / NUM_RUNS
    #print(f"probability of fixation for first allele: {probability_of_first_allele_fixation:.4f}")
    difference = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
    print(f"{difference:.4f} = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation")


    if fixed_generations:
        average_fixation_time = np.mean(fixed_generations)
        #print(f"avg time to fixation: {average_fixation_time:.2f} generations")
    else:
        pass # Added 'pass' to fix IndentationError
        #print("no fixation")

    s = fitness1 / fitness2 - 1
    c = 1
    time_to_fixation_pred1 = 2 * np.log(c * INITIAL_ALLELE_COUNTS.sum() - 1) / s

    N = size1 + size2

    p = size1 / N
    q = size2 / N

    mean_fitness = p * fitness1 + q * fitness2

    Vw = p * q * (fitness1 - fitness2)**2 / mean_fitness**2

    Ne = N / (1 + Vw)

    time_to_fixation_pred2 = 2 * np.log(c * Ne - 1) / s

    if INITIAL_ALLELE_COUNTS[0] == 1 and s > 0:
      if s < 1 / Ne:
            print(f"{time_to_fixation_pred2:.2f} generations = average_fixation_time - time_to_fixation_pred2")
      else:
            print(f"{time_to_fixation_pred1:.2f} generations = average_fixation_time - time_to_fixation_pred1 (assuming strong selection)")




if __name__ == '__main__':

    #BENEFICIAL OR NEUTRAL

    # one initial allele

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_2, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_2, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_3, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_3, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_4, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_4, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_5, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_5, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_6, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_6, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_7, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_7, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_8, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_8, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_9, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_9, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_10, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_10, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_11, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_11, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_12, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_12, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    # more than one initial allele

    INITIAL_ALLELE_COUNTS = np.array([2, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([3, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([4, 20])
    FITNESS_FUNCTIONS = [constant_fitness_2, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([5, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_2, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([6, 20])
    FITNESS_FUNCTIONS = [constant_fitness_3, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([7, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_3, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([8, 20])
    FITNESS_FUNCTIONS = [constant_fitness_4, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([90, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_4, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([10, 20])
    FITNESS_FUNCTIONS = [constant_fitness_5, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([700, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_5, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([19, 20])
    FITNESS_FUNCTIONS = [constant_fitness_6, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([100, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_6, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([11, 20])
    FITNESS_FUNCTIONS = [constant_fitness_7, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([9999, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_7, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([3, 20])
    FITNESS_FUNCTIONS = [constant_fitness_8, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([400, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_8, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([3, 20])
    FITNESS_FUNCTIONS = [constant_fitness_9, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([90, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_9, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([12, 20])
    FITNESS_FUNCTIONS = [constant_fitness_10, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([4, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_10, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([3, 20])
    FITNESS_FUNCTIONS = [constant_fitness_11, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([7, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_11, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([2, 20])
    FITNESS_FUNCTIONS = [constant_fitness_12, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([900, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_12, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    # DELETERIOUS

        # one initial allele

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_2, constant_fitness_3]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_2, constant_fitness_3]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_2]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_2]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_3]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_3]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_4]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_4]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_5]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_5]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_6]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_6]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_7]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_7]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_8]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_8]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_9]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_9]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_10]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_10]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_11]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_11]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_12]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([1, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_12]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    # more than one initial allele

    INITIAL_ALLELE_COUNTS = np.array([2, 20])
    FITNESS_FUNCTIONS = [constant_fitness_2, constant_fitness_3]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([3, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_2, constant_fitness_3]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([4, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_2]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([5, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_2]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([6, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_3]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([7, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_3]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([8, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_4]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([90, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_4]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([10, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_5]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([700, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_5]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([19, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_6]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([100, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_6]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([11, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_7]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([9999, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_7]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([3, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_8]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([400, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_8]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([3, 20])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_9]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([90, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_1, constant_fitness_9]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([12, 20])
    FITNESS_FUNCTIONS = [constant_fitness_10, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([4, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_10, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([3, 20])
    FITNESS_FUNCTIONS = [constant_fitness_11, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([7, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_11, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([2, 20])
    FITNESS_FUNCTIONS = [constant_fitness_12, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

    INITIAL_ALLELE_COUNTS = np.array([900, 1000])
    FITNESS_FUNCTIONS = [constant_fitness_12, constant_fitness_1]
    NUM_RUNS = 4000
    run_everything(INITIAL_ALLELE_COUNTS, FITNESS_FUNCTIONS, NUM_RUNS)

/n
INITIAL_ALLELE_COUNTS = [ 1 20]
FITNESS_FUNCTIONS = [<function constant_fitness_1 at 0x7c6a5c68b600>, <function constant_fitness_1 at 0x7c6a5c68b600>]
0.0069 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
/n
INITIAL_ALLELE_COUNTS = [   1 1000]
FITNESS_FUNCTIONS = [<function constant_fitness_1 at 0x7c6a5c68b600>, <function constant_fitness_1 at 0x7c6a5c68b600>]


/tmp/ipykernel_538/4210982700.py:244: RuntimeWarning: divide by zero encountered in scalar divide
  time_to_fixation_pred1 = 2 * np.log(c * INITIAL_ALLELE_COUNTS.sum() - 1) / s
/tmp/ipykernel_538/4210982700.py:257: RuntimeWarning: divide by zero encountered in scalar divide
  time_to_fixation_pred2 = 2 * np.log(c * Ne - 1) / s


0.0015 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
/n
INITIAL_ALLELE_COUNTS = [ 1 20]
FITNESS_FUNCTIONS = [<function constant_fitness_2 at 0x7c6a54a7e200>, <function constant_fitness_1 at 0x7c6a5c68b600>]
-0.0093 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
5.99 generations = average_fixation_time - time_to_fixation_pred1 (assuming strong selection)
/n
INITIAL_ALLELE_COUNTS = [   1 1000]
FITNESS_FUNCTIONS = [<function constant_fitness_2 at 0x7c6a54a7e200>, <function constant_fitness_1 at 0x7c6a5c68b600>]
-0.0021 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
13.82 generations = average_fixation_time - time_to_fixation_pred1 (assuming strong selection)
/n
INITIAL_ALLELE_COUNTS = [ 1 20]
FITNESS_FUNCTIONS = [<function constant_fitness_3 at 0x7c6a52ad32e0>, <function constant_fitness_1 at 0x7c6a5c68b600>]
-0.0087 = probability_of_first_allele_fixation - expected_pro

/tmp/ipykernel_538/4210982700.py:213: RuntimeWarning: divide by zero encountered in log
  malthusian_selection_coefficient = np.log(wrightian_selection_coefficient+1)
/tmp/ipykernel_538/4210982700.py:214: RuntimeWarning: invalid value encountered in log
  malthusian_fitness = np.log(1+malthusian_selection_coefficient)


0.9990 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
/n
INITIAL_ALLELE_COUNTS = [   5 1000]
FITNESS_FUNCTIONS = [<function constant_fitness_2 at 0x7c6a54a7e200>, <function constant_fitness_1 at 0x7c6a5c68b600>]
0.9990 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
/n
INITIAL_ALLELE_COUNTS = [ 6 20]
FITNESS_FUNCTIONS = [<function constant_fitness_3 at 0x7c6a52ad32e0>, <function constant_fitness_1 at 0x7c6a5c68b600>]


/tmp/ipykernel_538/4210982700.py:213: RuntimeWarning: invalid value encountered in log
  malthusian_selection_coefficient = np.log(wrightian_selection_coefficient+1)


1.0000 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
/n
INITIAL_ALLELE_COUNTS = [   7 1000]
FITNESS_FUNCTIONS = [<function constant_fitness_3 at 0x7c6a52ad32e0>, <function constant_fitness_1 at 0x7c6a5c68b600>]
1.0000 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
/n
INITIAL_ALLELE_COUNTS = [ 8 20]
FITNESS_FUNCTIONS = [<function constant_fitness_4 at 0x7c6a6bf0ede0>, <function constant_fitness_1 at 0x7c6a5c68b600>]
1.0000 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
/n
INITIAL_ALLELE_COUNTS = [  90 1000]
FITNESS_FUNCTIONS = [<function constant_fitness_4 at 0x7c6a6bf0ede0>, <function constant_fitness_1 at 0x7c6a5c68b600>]
1.0000 = probability_of_first_allele_fixation - expected_probability_of_first_allele_fixation
/n
INITIAL_ALLELE_COUNTS = [10 20]
FITNESS_FUNCTIONS = [<function constant_fitness_5 at 0x7c6a550b2f20>, <function constant_fitness_1 at 0x7c6a5c68b600>]
